### Model Training


### Step 1: Objective

The objective of this notebook is to train and evaluate machine learning models for predicting Total Revenue using the feature-engineered dataset.

The notebook follows these steps:

- Load the feature-engineered dataset
- Split the data into training and testing datasets
- Build a reusable Spark ML Pipeline
- Train a baseline Linear Regression model
- Evaluate model performance using RMSE, MAE, and R²
- Compare with advanced models in later sections

#### Step 2: Load the Feature Dataset

In [0]:
feature_df = spark.read.table('retail_project.gold.ml_model_sales_data')

display(feature_df)

In [0]:
# Check the schema
feature_df.printSchema()

In [0]:
#Checking the number of rows and columns (shape of the Data)
print("Rows :", feature_df.count())
print("Columns :", len(feature_df.columns))

#### Step 3: Train/Test Split

In [0]:
# Spliting the Dataset in Train and Test

train_df, test_df = feature_df.randomSplit([0.8, 0.2], seed=42)

# Checking the shape of the Train and Test Dataset
print("Training Dataset :", train_df.count(), len(train_df.columns))
print("Testing Dataset :", test_df.count(), len(test_df.columns))

#### Step 4: Recreate the Preprocessing Pipeline

In [0]:
# Categorical Columns

categorical_cols = [
    "product_id",
    "category",
    "gender",
    "city",
    "state"
]

# Numerical Columns

numeric_cols = [
    "Month",
    "Quarter",
    "WeekOfYear",
    "DayOfWeek",
    "DayOfMonth",
    "Total_quantity",
    "Max_price"
]

##### StringIndexer

In [0]:
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(
        inputCol=col_name,
        outputCol=col_name + '_index',
        handleInvalid='keep'
    )
    for col_name in categorical_cols
]

##### OneHotEncoding

In [0]:
from pyspark.ml.feature import OneHotEncoder

encoders = [
    OneHotEncoder(
        inputCols = [c + '_index' for c in categorical_cols],
        outputCols = [c + '_encoded' for c in categorical_cols]
    )
]

##### Vector Assembler

In [0]:
from pyspark.ml.  feature import VectorAssembler

assembler = VectorAssembler(
    inputCols= [
        'product_id_encoded',
        'category_encoded',
        'gender_encoded',
        'city_encoded',
        'state_encoded',
        *numeric_cols
    ],
    outputCol='features'
)

#### Step 5: Create the Baseline Model (Linear Regression)

- **Why Linear Regression first?**

-  Before training complex models like Random Forest, we need a baseline.

In [0]:
from pyspark.ml.regression import LinearRegression

lr_model = LinearRegression(
    featuresCol='features',
    labelCol='Total_revenue',
    predictionCol='prediction',
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

| Parameter                    | Meaning                                           | Why                                             |
| ---------------------------- | ------------------------------------------------- | ----------------------------------------------- |
| `featuresCol="features"`     | Input feature vector created by `VectorAssembler` | Required by Spark ML                            |
| `labelCol="Total_revenue"`   | Target variable                                   | What we're predicting                           |
| `predictionCol="prediction"` | Output prediction column                          | Default output                                  |
| `maxIter=100`                | Maximum optimization iterations                   | Usually sufficient for convergence              |
| `regParam=0.0`               | Regularization strength                           | `0.0` means ordinary Linear Regression          |
| `elasticNetParam=0.0`        | Type of regularization                            | `0.0` = Ridge style when regularization is used |


#### Step 6: Build the Complete Pipeline for Linerar Regression Model

In [0]:
from pyspark.ml import Pipeline

lr_pipeline = Pipeline(stages= indexers + encoders + [assembler, lr_model])

#### Step 7: Train the Model

In [0]:
# Note: If you get ML_CACHE_SIZE_OVERFLOW_EXCEPTION:
# This occurs when training multiple large models in one session.
# Workaround: Detach and re-attach, then re-run cells 1-22 before running this cell.
# Or: Train only one model per session (skip either this cell or Cell 40).

lr_pipeline_model = lr_pipeline.fit(train_df)

#### Step 8: Make Predictions

In [0]:
lr_predictions = lr_pipeline_model.transform(test_df)


In [0]:
# Verify the ouput

display(lr_predictions.select('Total_revenue', 'prediction'))

#### Step 9: Evaluate the Model

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

metrics = ['mse', 'rmse', 'mae', 'r2']

for metric in metrics:
    evaluator = RegressionEvaluator(
        labelCol= 'Total_revenue',
        predictionCol= 'prediction',
        metricName =  metric
    )

    score = evaluator.evaluate(lr_predictions)
    print(f' Evaluation Score for {metric.upper()} : {score}')
    


| Metric   |              Value | Interpretation                                                                                          |
| -------- | -----------------: | ------------------------------------------------------------------------------------------------------- |
| **MSE**  | **353,924,676.91** | Squared error. Mainly used for optimization; the unit is squared, so it's harder to interpret directly. |
| **RMSE** |      **18,812.89** | Average prediction error is about **₹18.8K**. Lower is better.                                          |
| **MAE**  |      **14,425.90** | On average, predictions differ from the actual revenue by about **₹14.4K**.                             |
| **R²**   |         **0.8942** | **Excellent.** The model explains approximately **89.4%** of the variation in `Total_revenue`.          |


### Baseline Model Results

A Linear Regression model was trained as the baseline regression model.

#### Evaluation Metrics

- MSE: 353,924,676.91
- RMSE: 18,812.89
- MAE: 14,425.90
- R²: 0.8942

#### Observations

- The model explains approximately 89.4% of the variance in Total Revenue.
- The average prediction error is around ₹14.4K (MAE).
- Some predictions are negative, indicating that Linear Regression does not fully capture the nonlinear relationships in the retail sales data.
- More advanced tree-based models such as Decision Tree and Random Forest will be evaluated to improve predictive performance.

### Decision Tree Regressor

### Step 1: Objective

- Train a Decision Tree Regressor using the same feature engineering pipeline and compare its performance with the Linear Regression baseline.

- This model is capable of learning non-linear relationships between the input features and the target variable.

#### Step 2: Import & Create the Decision Tree Model

In [0]:
from pyspark.ml.regression import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(
    featuresCol= 'features',
    labelCol= 'Total_revenue',
    predictionCol= 'prediction',
    maxDepth=10,
    minInstancesPerNode= 5,
    seed=42
)

| Parameter             | Value | Reason                                                           |
| --------------------- | ----: | ---------------------------------------------------------------- |
| `maxDepth`            |    10 | Controls tree complexity. Prevents an excessively deep tree.     |
| `minInstancesPerNode` |     5 | Reduces overfitting by requiring at least 5 rows in a leaf node. |
| `seed`                |    42 | Makes the results reproducible.                                  |


### Step 3: Build the Pipeline

In [0]:
del lr_model, lr_pipeline_model   # any fitted model / pipeline objects
import gc
gc.collect()

In [0]:
from pyspark.ml import Pipeline

dt_pipeline = Pipeline(stages=indexers + encoders + [assembler, dt_model])

#### Step 4: Train the Model and make Prediction for DT Model

In [0]:
# Workaround: Spark Connect ML cache limit (1GB)
# The cache persists on the server even after deleting Python references.
# Solution: Restart Python → re-run Cells 1-38 (skip Cell 23) → run this cell

dt_pipeline_model = dt_pipeline.fit(train_df)

dt_predictions = dt_pipeline_model.transform(test_df)

Step 5: Make Predictions for DT Model

In [0]:
display(
    dt_predictions.select(
        "Total_revenue",
        "prediction"
    )
)

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

metrics = ["mse", "rmse", "mae", "r2"]

for metric in metrics:
    evaluator = RegressionEvaluator(
        labelCol="Total_revenue",
        predictionCol="prediction",
        metricName=metric
    )

    score = evaluator.evaluate(dt_predictions)

    print(f"Evalution Score for {metric.upper()} : {score}")

#### **Decision Tree Model – Inference**

* **R² = 0.9914** → The model explains about **99.14% of the variation in revenue**, indicating an excellent fit.
* **RMSE = 5,364** → Predictions typically differ from actual revenue by around **₹5.36K**.
* **MAE = 3,685** → The average prediction error is approximately **₹3.68K**.
* **MSE = 28.77M** → This is mainly influenced by larger prediction errors.

#### **Overall**

✅ The **Decision Tree model performs very well**, with high predictive accuracy and relatively low errors.
⚠️ However, the **very high R² (99.14%)** should be checked for possible **overfitting or data leakage**, especially because some features may be directly related to `Total_revenue`. will do model Experiments


### MLflow Experiment for Decision Tree Models

#### Step 1 — Create the MLflow Experiment

In [0]:
import mlflow

experiment_name = "/Users/deepakraj140498@gmail.com/retail_sales_ml_experiment"

mlflow.set_experiment(experiment_name)

print("MLflow experiment created/selected successfully")

In [0]:
with mlflow.start_run(run_name="Decision_Tree_Baseline"):

    # Train
    dt_pipeline_model = dt_pipeline.fit(train_df)

    # Predict
    dt_predictions = dt_pipeline_model.transform(test_df)

    # Evaluate
    dt_mse = evaluator_mse.evaluate(dt_predictions)
    dt_rmse = evaluator_rmse.evaluate(dt_predictions)
    dt_mae = evaluator_mae.evaluate(dt_predictions)
    dt_r2 = evaluator_r2.evaluate(dt_predictions)

    # Parameters
    mlflow.log_param("model", "Decision Tree")
    mlflow.log_param("maxDepth", 10)
    mlflow.log_param("minInstancesPerNode", 5)
    mlflow.log_param("seed", 42)

    # Metrics
    mlflow.log_metric("MSE", dt_mse)
    mlflow.log_metric("RMSE", dt_rmse)
    mlflow.log_metric("MAE", dt_mae)
    mlflow.log_metric("R2", dt_r2)

    print("MSE :", dt_mse)
    print("RMSE:", dt_rmse)
    print("MAE :", dt_mae)
    print("R2  :", dt_r2)

In [0]:
print("dt_pipeline_model" in globals())
print("lr_pipeline_model" in globals())
print("dt_pipeline" in globals())

In [0]:
#Step 5 — Now test preprocessing ONLY preprocessing and then train the model on the preprocessed data.
from pyspark.ml import Pipeline

preprocess_pipeline = Pipeline(
    stages=indexers + encoders + [assembler]
)

preprocess_model = preprocess_pipeline.fit(train_df)

processed_train = preprocess_model.transform(train_df)

display(
    processed_train.select(
        "features",
        "Total_revenue"
    ).limit(5)
)

Next step: train Decision Tree separately

In [0]:
# do not recreate the preprocessing pipeline and do not use dt_pipeline.fit(train_df) now.
from pyspark.ml.regression import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="Total_revenue",
    predictionCol="prediction",
    maxDepth=10,
    minInstancesPerNode=5,
    seed=42
)

dt_model_fitted = dt_model.fit(processed_train)

In [0]:
# Then transform the test data:

processed_test = preprocess_model.transform(test_df)

In [0]:
dt_predictions = dt_model_fitted.transform(processed_test)

display(
    dt_predictions.select(
        "Total_revenue",
        "prediction"
    ).limit(10)
)

In [0]:
#Now let's evaluate this model
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_mse = RegressionEvaluator(
    labelCol="Total_revenue",
    predictionCol="prediction",
    metricName="mse"
)

evaluator_rmse = RegressionEvaluator(
    labelCol="Total_revenue",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="Total_revenue",
    predictionCol="prediction",
    metricName="mae"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="Total_revenue",
    predictionCol="prediction",
    metricName="r2"
)

dt_mse = evaluator_mse.evaluate(dt_predictions)
dt_rmse = evaluator_rmse.evaluate(dt_predictions)
dt_mae = evaluator_mae.evaluate(dt_predictions)
dt_r2 = evaluator_r2.evaluate(dt_predictions)

print("Decision Tree Evaluation")
print("------------------------")
print("MSE  :", dt_mse)
print("RMSE :", dt_rmse)
print("MAE  :", dt_mae)
print("R2   :", dt_r2)

In [0]:
#First create/select the experiment

import mlflow

dt_experiment_name = "/Users/deepakraj140498@gmail.com/retail_sales_ml_experiment"

mlflow.set_experiment(dt_experiment_name)

In [0]:
#Then we will start runing the experiment

with mlflow.start_run(run_name="Decision_Tree_Baseline"):

    # Parameters
    mlflow.log_param("model", "Decision Tree")
    mlflow.log_param("maxDepth", 10)
    mlflow.log_param("minInstancesPerNode", 5)
    mlflow.log_param("seed", 42)

    # Metrics
    mlflow.log_metric("MSE", dt_mse)
    mlflow.log_metric("RMSE", dt_rmse)
    mlflow.log_metric("MAE", dt_mae)
    mlflow.log_metric("R2", dt_r2)

    print("Decision Tree experiment logged successfully.")

Great. 👍 Our Decision Tree baseline is now successfully tracked in MLflow.

Now the next step is not Random Forest yet. We should first do a small but important feature-leakage check, because your R² of 0.9914 is extremely high.

In [0]:
#Step 2 — Check the features used for prediction

print("Target column:")
print("Total_revenue")

print("\nCategorical columns:")
print(categorical_cols)

print("\nNumerical columns:")
print(numeric_cols)

In [0]:
#Then we run the feature importance analysis
feature_df.select(
    *categorical_cols,
    *numeric_cols,
    "Total_revenue"
).printSchema()

In [0]:
print(numeric_cols)

### Next: Random Forest Model

In [0]:
# Random Forest Regression Model
from pyspark.ml.regression import RandomForestRegressor

rf_model = RandomForestRegressor(
    featuresCol="features",
    labelCol="Total_revenue",
    predictionCol="prediction",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_model_fitted = rf_model.fit(processed_train)

In [0]:
# Evaluate the model
rf_predictions = rf_model_fitted.transform(processed_test)

display(
    rf_predictions.select(
        "Total_revenue",
        "prediction"
    ).limit(10)
)

Perfect. ✅ Random Forest has also trained successfully.

The predictions look reasonable, so now let's evaluate Random Forest using the same four metrics as Linear Regression and Decision Tree.

In [0]:
# Step 1 — Evaluate Random Forest using the evalution metrics

rf_mse = evaluator_mse.evaluate(rf_predictions)
rf_rmse = evaluator_rmse.evaluate(rf_predictions)
rf_mae = evaluator_mae.evaluate(rf_predictions)
rf_r2 = evaluator_r2.evaluate(rf_predictions)

print("Random Forest Evaluation")
print("------------------------")
print("MSE  :", rf_mse)
print("RMSE :", rf_rmse)
print("MAE  :", rf_mae)
print("R2   :", rf_r2)

#### **Random Forest Model – Inference**

* **R² = 0.9755** → The model explains approximately **97.55% of the variation in revenue**.
* **RMSE = 9,057** → Predictions are off by around **₹9.06K on average** in terms of larger errors.
* **MAE = 6,567** → Average prediction error is approximately **₹6.57K**.
* **MSE = 82.03M** → Higher than the Decision Tree model.

#### **Overall**

✅ Random Forest performs **very well**, with **97.55% R²**.
📊 However, the **Decision Tree performs better** on this dataset because it has **lower MSE, RMSE, and MAE** and a higher R² (**99.14% vs 97.55%**).

⚠️ The high R² of both models should still be checked for **overfitting/data leakage** before selecting the final model.


#### Next: Log Random Forest in MLflow

### **Random Forest Model – Inference**

* **R² = 0.9755** → The model explains approximately **97.55% of the variation in revenue**.
* **RMSE = 9,057** → Predictions are off by around **₹9.06K on average** in terms of larger errors.
* **MAE = 6,567** → Average prediction error is approximately **₹6.57K**.
* **MSE = 82.03M** → Higher than the Decision Tree model.

### **Overall**

✅ Random Forest performs **very well**, with **97.55% R²**.
📊 However, the **Decision Tree performs better** on this dataset because it has **lower MSE, RMSE, and MAE** and a higher R² (**99.14% vs 97.55%**).

⚠️ The high R² of both models should still be checked for **overfitting/data leakage** before selecting the final model.


### Next: Log Random Forest in MLflow

In [0]:
# Step 2 — Log the model to MLflow Experiment
with mlflow.start_run(run_name="Random_Forest_Baseline"):

    # Parameters
    mlflow.log_param("model", "Random Forest")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 10)
    mlflow.log_param("seed", 42)

    # Metrics
    mlflow.log_metric("MSE", rf_mse)
    mlflow.log_metric("RMSE", rf_rmse)
    mlflow.log_metric("MAE", rf_mae)
    mlflow.log_metric("R2", rf_r2)

    print("Random Forest experiment logged successfully.")

In [0]:
#Your MLflow experiment should now contain:

'''
retail_sales_ml_experiment
│
├── Decision_Tree_Baseline
│
└── Random_Forest_Baseline
'''

'''
retail_sales_ml_experiment
│
- ├── Decision_Tree_Baseline
│     R² = 0.9914
│
└── Random_Forest_Baseline
      R² = 0.9755

      '''

In [0]:
'''
retail_sales_ml_experiment
│
├── Decision_Tree_Baseline
│     R² = 0.9914
│
└── Random_Forest_Baseline
      R² = 0.9755
 '''

### Next step: XGBoost Model 🚀

In [0]:
# We'll follow the same safe approach that worked for Decision Tree and Random Forest:

'''
processed_train
      ↓
XGBoost Regressor
      ↓
predictions
      ↓
MSE / RMSE / MAE / R²
      ↓
MLflow
'''

We will not create another full preprocessing pipeline.

Before I give you the XGBoost code, one important point: because you're using Databricks Free Edition/Serverless and already hit the Spark Connect ML cache limit, we'll use a single XGBoost model first, evaluate it, and only then decide whether hyperparameter experiments are worthwhile.

In [0]:
%pip install xgboost

In [0]:
# Train XGBoost Regressor we got error while doing all at once so below splited and done
from xgboost.spark import SparkXGBRegressor

xgb_model = SparkXGBRegressor(
    features_col="features",
    label_col="Total_revenue",
    prediction_col="prediction",
    num_workers=1,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb_model_fitted = xgb_model.fit(processed_train)

In [0]:
from xgboost.spark import SparkXGBRegressor


In [0]:
xgb_model = SparkXGBRegressor(
    features_col="features",
    label_col="Total_revenue",
    prediction_col="prediction",
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

In [0]:
xgb_model_fitted = xgb_model.fit(processed_train)

In [0]:
xgb_model.fit(processed_train)

In [0]:
# Step 1 — Convert Spark features to NumPy Pandas DataFrames
import numpy as np

train_pd = processed_train.select(
    "features",
    "Total_revenue"
).toPandas()

test_pd = processed_test.select(
    "features",
    "Total_revenue"
).toPandas()

In [0]:
# X_train and X_test are NumPy arrays of shape (n_samples, n_features)

X_train = np.array(
    train_pd["features"].apply(lambda x: x.toArray()).tolist()
)

y_train = train_pd["Total_revenue"].values

X_test = np.array(
    test_pd["features"].apply(lambda x: x.toArray()).tolist()
)

y_test = test_pd["Total_revenue"].values

In [0]:
#Shape of the NumPy arrays X_train, y_train and X_test and y_test
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

In [0]:
# Step 2 — Train normal XGBoost
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train, y_train)

In [0]:
#Step 3 — Predict
xgb_pred = xgb_model.predict(X_test)

In [0]:
# Step 4 Evaluate the model 

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(xgb_mse)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost Evaluation")
print("------------------")
print("MSE  :", xgb_mse)
print("RMSE :", xgb_rmse)
print("MAE  :", xgb_mae)
print("R2   :", xgb_r2)

### Model Comparison 

| Model             |       RMSE ↓ |        MAE ↓ |        R² ↑ | Rank |
| ----------------- | -----------: | -----------: | ----------: | ---: |
| Linear Regression |    18,812.89 |    14,425.90 |      0.8942 |    4 |
| Decision Tree     |     5,364.00 |     3,684.71 |      0.9914 |    2 |
| Random Forest     |     9,057.15 |     6,566.77 |      0.9755 |    3 |
| **XGBoost**       | **4,556.37** | **3,233.61** | **0.99379** | 🥇 1 |


### Next Step: Log XGBoost in MLflow

In [0]:
with mlflow.start_run(run_name="XGBoost_Baseline"):

    # Parameters
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("random_state", 42)

    # Metrics
    mlflow.log_metric("MSE", xgb_mse)
    mlflow.log_metric("RMSE", xgb_rmse)
    mlflow.log_metric("MAE", xgb_mae)
    mlflow.log_metric("R2", xgb_r2)

    print("XGBoost experiment logged successfully.")

In [0]:
# Your MLflow experiment will now contain:
''''
retail_sales_ml_experiment
│
├── Decision_Tree_Baseline
│
├── Random_Forest_Baseline
│
└── XGBoost_Baseline
''''''

### Step 1 — First experiment

- We'll change only n_estimators and keep everything else the same.

In [0]:
from xgboost import XGBRegressor
import mlflow

with mlflow.start_run(run_name="XGBoost_Exp_1"):

    xgb_exp1 = XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        objective="reg:squarederror"
    )

    xgb_exp1.fit(X_train, y_train)

    pred_exp1 = xgb_exp1.predict(X_test)

    exp1_mse = mean_squared_error(y_test, pred_exp1)
    exp1_rmse = np.sqrt(exp1_mse)
    exp1_mae = mean_absolute_error(y_test, pred_exp1)
    exp1_r2 = r2_score(y_test, pred_exp1)

    # Parameters
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("random_state", 42)

    # Metrics
    mlflow.log_metric("MSE", exp1_mse)
    mlflow.log_metric("RMSE", exp1_rmse)
    mlflow.log_metric("MAE", exp1_mae)
    mlflow.log_metric("R2", exp1_r2)

    print("XGBoost Experiment 1")
    print("-------------------")
    print("MSE  :", exp1_mse)
    print("RMSE :", exp1_rmse)
    print("MAE  :", exp1_mae)
    print("R2   :", exp1_r2)

| Model            |       RMSE ↓ |        MAE ↓ |         R² ↑ |
| ---------------- | -----------: | -----------: | -----------: |
| XGBoost Baseline | **4,556.37** |     3,233.61 | **0.993792** |
| XGBoost Exp 1    |     4,608.93 | **3,233.15** |     0.993648 |


#### Experiment 2 — Change max_depth

In [0]:
from xgboost import XGBRegressor
import mlflow

with mlflow.start_run(run_name="XGBoost_Exp_2"):

    xgb_exp2 = XGBRegressor(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        objective="reg:squarederror"
    )

    xgb_exp2.fit(X_train, y_train)

    pred_exp2 = xgb_exp2.predict(X_test)

    exp2_mse = mean_squared_error(y_test, pred_exp2)
    exp2_rmse = np.sqrt(exp2_mse)
    exp2_mae = mean_absolute_error(y_test, pred_exp2)
    exp2_r2 = r2_score(y_test, pred_exp2)

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("MSE", exp2_mse)
    mlflow.log_metric("RMSE", exp2_rmse)
    mlflow.log_metric("MAE", exp2_mae)
    mlflow.log_metric("R2", exp2_r2)

    print("XGBoost Experiment 2")
    print("-------------------")
    print("MSE  :", exp2_mse)
    print("RMSE :", exp2_rmse)
    print("MAE  :", exp2_mae)
    print("R2   :", exp2_r2)

####  Current XGBoost comparison
| Run         | n_estimators | max_depth | learning_rate |      RMSE ↓ |       MAE ↓ |         R² ↑ |
| ----------- | -----------: | --------: | ------------: | ----------: | ----------: | -----------: |
| 🏆 Baseline |          100 |         6 |           0.1 | **4556.37** |     3233.61 | **0.993792** |
| Exp 1       |          200 |         6 |           0.1 |     4608.93 | **3233.15** |     0.993648 |
| Exp 2       |          100 |         4 |           0.1 |     4558.13 |     3304.16 |     0.993787 |


Simple inference

Baseline is still the best.

#### Experiment 3 — Learning Rate Tuning

In [0]:
'''
n_estimators = 100
max_depth = 6
learning_rate = 0.05
'''
#What we're learning
# learning_rate controls how strongly each new tree contributes to the final model.
''''
0.10  → baseline
0.05  → Experiment 3

'''

In [0]:
with mlflow.start_run(run_name="XGBoost_Exp_3"):

    xgb_exp3 = XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        objective="reg:squarederror"
    )

    xgb_exp3.fit(X_train, y_train)

    pred_exp3 = xgb_exp3.predict(X_test)

    exp3_mse = mean_squared_error(y_test, pred_exp3)
    exp3_rmse = np.sqrt(exp3_mse)
    exp3_mae = mean_absolute_error(y_test, pred_exp3)
    exp3_r2 = r2_score(y_test, pred_exp3)

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("MSE", exp3_mse)
    mlflow.log_metric("RMSE", exp3_rmse)
    mlflow.log_metric("MAE", exp3_mae)
    mlflow.log_metric("R2", exp3_r2)

    print("XGBoost Experiment 3")
    print("-------------------")
    print("MSE  :", exp3_mse)
    print("RMSE :", exp3_rmse)
    print("MAE  :", exp3_mae)
    print("R2   :", exp3_r2)

#### Current comparison

| Run          | n_estimators | max_depth | learning_rate |      RMSE ↓ |       MAE ↓ |         R² ↑ |
| ------------ | -----------: | --------: | ------------: | ----------: | ----------: | -----------: |
| 🥇 **Exp 3** |          100 |         6 |      **0.05** | **4545.73** |     3250.37 | **0.993821** |
| Baseline     |          100 |         6 |          0.10 |     4556.37 | **3233.61** |     0.993792 |
| Exp 2        |          100 |         4 |          0.10 |     4558.13 |     3304.16 |     0.993787 |
| Exp 1        |          200 |         6 |          0.10 |     4608.93 |     3233.15 |     0.993648 |


### Experiment 4 — subsample (adding one mor parameter tuning)

In [0]:
'''
Why subsample?

It controls the fraction of training rows used by each boosting round.
This is a technique to prevent overfitting

subsample = 1.0
    ↓
100% of rows

subsample = 0.8
    ↓
80% of rows

'''

In [0]:
with mlflow.start_run(run_name="XGBoost_Exp_4"):

    xgb_exp4 = XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42,
        objective="reg:squarederror"
    )

    xgb_exp4.fit(X_train, y_train)

    pred_exp4 = xgb_exp4.predict(X_test)

    exp4_mse = mean_squared_error(y_test, pred_exp4)
    exp4_rmse = np.sqrt(exp4_mse)
    exp4_mae = mean_absolute_error(y_test, pred_exp4)
    exp4_r2 = r2_score(y_test, pred_exp4)

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("MSE", exp4_mse)
    mlflow.log_metric("RMSE", exp4_rmse)
    mlflow.log_metric("MAE", exp4_mae)
    mlflow.log_metric("R2", exp4_r2)

    print("XGBoost Experiment 4")
    print("-------------------")
    print("MSE  :", exp4_mse)
    print("RMSE :", exp4_rmse)
    print("MAE  :", exp4_mae)
    print("R2   :", exp4_r2)

#### XGBoost experiment comparison

| Run          | n_estimators | max_depth | learning_rate | subsample |      RMSE ↓ |       MAE ↓ |         R² ↑ |
| ------------ | -----------: | --------: | ------------: | --------: | ----------: | ----------: | -----------: |
| Baseline     |          100 |         6 |          0.10 |       1.0 |     4556.37 |     3233.61 |     0.993792 |
| Exp 1        |          200 |         6 |          0.10 |       1.0 |     4608.93 |     3233.15 |     0.993648 |
| Exp 2        |          100 |         4 |          0.10 |       1.0 |     4558.13 |     3304.16 |     0.993787 |
| Exp 3        |          100 |         6 |          0.05 |       1.0 |     4545.73 |     3250.37 |     0.993821 |
| 🥇 **Exp 4** |      **100** |     **6** |      **0.05** |   **0.8** | **4481.25** | **3217.56** | **0.993995** |


#### Simple inference

Experiment 4 improved all three important metrics:

- RMSE: 4545.73 → 4481.25 ✅
- MAE: 3250.37 → 3217.56 ✅
- R²: 0.993821 → 0.993995 ✅

#### One more experiment using `colsample_bytree` - Final Experiment

- Let's do one final controlled experiment with `colsample_bytree`.


        - colsample_bytree = 0.8
This will test whether using 80% of the features for each tree improves generalization.

In [0]:
with mlflow.start_run(run_name="XGBoost_Exp_5"):

    xgb_exp5 = XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror"
    )

    xgb_exp5.fit(X_train, y_train)

    pred_exp5 = xgb_exp5.predict(X_test)

    exp5_mse = mean_squared_error(y_test, pred_exp5)
    exp5_rmse = np.sqrt(exp5_mse)
    exp5_mae = mean_absolute_error(y_test, pred_exp5)
    exp5_r2 = r2_score(y_test, pred_exp5)

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("MSE", exp5_mse)
    mlflow.log_metric("RMSE", exp5_rmse)
    mlflow.log_metric("MAE", exp5_mae)
    mlflow.log_metric("R2", exp5_r2)

    print("XGBoost Experiment 5")
    print("-------------------")
    print("MSE  :", exp5_mse)
    print("RMSE :", exp5_rmse)
    print("MAE  :", exp5_mae)
    print("R2   :", exp5_r2)

### XGBoost Experiment Comparison

| Run          |       RMSE ↓ |        MAE ↓ |         R² ↑ | Result   |
| ------------ | -----------: | -----------: | -----------: | -------- |
| Baseline     |     4,556.37 |     3,233.61 |     0.993792 | Good     |
| Exp 1        |     4,608.93 |     3,233.15 |     0.993648 | ❌        |
| Exp 2        |     4,558.13 |     3,304.16 |     0.993787 | ❌        |
| Exp 3        |     4,545.73 |     3,250.37 |     0.993821 | Good     |
| 🥇 **Exp 4** | **4,481.25** | **3,217.56** | **0.993995** | **Best** |
| Exp 5        |     5,937.38 |     4,166.70 |     0.989459 | ❌        |


### Why Experiment 5 failed

We added:

```text
colsample_bytree = 0.8
```

to our best configuration.

But performance dropped substantially:

```text
RMSE: 4481 → 5937
R²:   0.9940 → 0.9895
```

So **we won't use `colsample_bytree=0.8`**.

---



#### 🏆 Final XGBoost configuration

For this round of experiments, **Experiment 4 is the winner**:

```text
n_estimators     = 100
max_depth        = 6
learning_rate    = 0.05
subsample        = 0.8
colsample_bytree = 1.0
```

Performance:

```text
MSE  = 20,081,582.68
RMSE = 4,481.25
MAE  = 3,217.56
R²   = 0.993995
```

And compared with your other models:

| Model             |       RMSE ↓ |       R² ↑ |
| ----------------- | -----------: | ---------: |
| Linear Regression |    18,812.89 |     0.8942 |
| Random Forest     |     9,057.15 |     0.9755 |
| Decision Tree     |     5,364.00 |     0.9914 |
| **XGBoost Exp 4** | **4,481.25** | **0.9940** |

So **XGBoost Experiment 4 is currently our final/best candidate.** 🏆


### Next stage: Model Registry

Now we have finished the **baseline + controlled hyperparameter experiments**.

Our workflow becomes:

```text
                    MLflow
                       │
       ┌───────────────┼───────────────┐
       ↓               ↓               ↓
 Linear Regression   Decision Tree   Random Forest
                                       │
                                       ↓
                                  XGBoost
                                       │
                              Hyperparameter
                               Experiments
                                       │
                                       ↓
                              🏆 Exp 4 Winner
                                       │
                                       ↓
                              Log Model Artifact
                                       │
                                       ↓
                              Model Registry
                                       │
                                       ↓
                              Model Version
                                       │
                                       ↓
                              Model Serving
```


#### But one important distinction

So far, our MLflow runs contain **parameters and metrics**.

We have **not yet registered the actual XGBoost model artifact**.

That's what we'll do next.

Because your winning model is a regular Python:

```python
XGBRegressor
```

we can log it directly with MLflow without involving Spark ML or `SparkXGBRegressor`. This also avoids the Spark Connect cache problem that caused trouble earlier.

**Next step: we'll create one clean final XGBoost model using Experiment 4's parameters, log the model artifact to MLflow, and then register that exact model.**


### Model Artifact

- A model artifact is the **saved version of your trained machine learning model**.

- We will take the winning Experiment 4 configuration:

        - n_estimators     = 100
        - max_depth        = 6
        - learning_rate    = 0.05
        - subsample        = 0.8

and create one **Final XGBoost model**, then **log that model itself into MLflow**.


#### Step 1 — Train the final model

In [0]:
from xgboost import XGBRegressor

final_xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    objective="reg:squarederror"
)

final_xgb_model.fit(X_train, y_train)

### Step 2 — Verify the final model

In [0]:
final_pred = final_xgb_model.predict(X_test)

final_mse = mean_squared_error(y_test, final_pred)
final_rmse = np.sqrt(final_mse)
final_mae = mean_absolute_error(y_test, final_pred)
final_r2 = r2_score(y_test, final_pred)

print("Final XGBoost Model")
print("------------------")
print("MSE  :", final_mse)
print("RMSE :", final_rmse)
print("MAE  :", final_mae)
print("R2   :", final_r2)

### Step 3 — Log the Actual model to MLflow ---> Model Artifact

In [0]:
# Now we do something different from the previous experiments.
'''
Previously: This is for Experiments 1-5
We logged the metrics and parameters to MLflow, but we did not log the model artifact.
This is because we were not using MLflow to manage our model deployment.


MLflow
 ├── parameters
 └── metrics

Now: This is for Model Artifact to log the model artifact to save our final model output
This is because we are using MLflow to manage our model deployment.

MLflow
 ├── parameters
 ├── metrics
 └── MODEL ARTIFACT  ← important
'''

In [0]:
import mlflow
import mlflow.xgboost

with mlflow.start_run(run_name="Final_XGBoost_Model") as run:

    mlflow.log_params({
        "model": "XGBoost",
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "random_state": 42
    })

    mlflow.log_metrics({
        "MSE": final_mse,
        "RMSE": final_rmse,
        "MAE": final_mae,
        "R2": final_r2
    })

    mlflow.xgboost.log_model(
        final_xgb_model,
        artifact_path="model"
    )

    print("Final XGBoost model logged successfully.")
    print("Run ID:", run.info.run_id)

What just happened?

We now have a real model artifact stored in MLflow.

Conceptually:

    -MLflow Experiment
       │
       └── Final_XGBoost_Model
              │
              ├── Parameters
              ├── Metrics
              └── model/
                    └── XGBoost model

#### **Inference**

* ✅ The **XGBoost model artifact was successfully logged** in MLflow.
* ⚠️ The `artifact_path` message is only a **deprecation warning**, not an error. Use `name="model"` for future runs.
* ⚠️ The important issue is the **missing model signature**.
* The signature defines the model's **expected input features and output format**.
* A signature is required when registering the model in **Unity Catalog**.
* Therefore, we should **log the model again with an `input_example`** so MLflow can automatically infer the signature.
* 🔄 **No need to retrain the model**; we only need to log the already-trained model correctly.

**Next step:** Re-log the existing XGBoost model with `name="model"` and `input_example`, then register it in Unity Catalog.


In [0]:
#Because your XGBoost model expects the 42-feature NumPy array, use one test row as the example.
# Input Example 

input_example = X_test[:1]

- X_test → contains the input features used for testing.
- [:1] → selects only the first row.
- input_example → stores that one row as a sample input.

In [0]:
# Final XGBoost model logged with signature.

import mlflow
import mlflow.xgboost

with mlflow.start_run(run_name="Final_XGBoost_Model_v2") as run:

    mlflow.log_params({
        "model": "XGBoost",
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "random_state": 42
    })

    mlflow.log_metrics({
        "MSE": final_mse,
        "RMSE": final_rmse,
        "MAE": final_mae,
        "R2": final_r2
    })

    mlflow.xgboost.log_model(
        final_xgb_model,
        name="model",
        input_example=X_test[:1]
    )

    print("Final XGBoost model logged with Signature.")
    print("Run ID:", run.info.run_id)

### 🚀 Next Step: Register the Model in Unity Catalog

- Now we'll take this exact MLflow model and register it.

- Before we do that, we need the 3-level Unity Catalog model name:

    - catalog.schema.model_name




In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS IN retail_project;

In [0]:
%sql
SHOW TABLES IN retail_project.gold;

#### Step 1 — Register your logged model

In [0]:
import mlflow

model_name = "retail_project.gold.retail_sales_xgboost"

model_uri = f"runs:/{run.info.run_id}/model" #99850011e20847a2a8096868c53da023

registered_model= mlflow.register_model(model_uri=model_uri, name=model_name)

print("Model Registered Successfully!")
print(f"Model Registered as {registered_model.name}")
print(f"Model Version is {registered_model.version}")

#### Step 2 — Verify registration Model

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

model_name = "retail_project.gold.retail_sales_xgboost"

# Get model details
model = client.get_registered_model(model_name)

print(f"Model Name: {model.name}")
print(f"Description: {model.description}")
print(f"Creation Timestamp: {model.creation_timestamp}")
print(f"Last Updated Timestamp: {model.last_updated_timestamp}")
print()

# List all versions
versions = client.search_model_versions(f"name='{model_name}'")

print(f"Total Versions: {len(versions)}")
print()

for version in versions:
    print(f"Version: {version.version}")
    print(f"  Status: {version.status}")
    print(f"  Run ID: {version.run_id}")
    print(f"  Creation Timestamp: {version.creation_timestamp}")
    print()

Before Model Serving, let's identify which version is your final/best model. Change Version name to Champion (alias name)

#### Step 1 — Assign Version 1 as champion

In [0]:
import mlflow
from mlflow import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name="retail_project.gold.retail_sales_xgboost",
    alias="champion",
    version=1
)

print("Champion alias assigned to Version 1")

#### Step 2 — Verify the alias

In [0]:
champion = client.get_model_version_by_alias(
    "retail_project.gold.retail_sales_xgboost",
    "champion"
)

print("Model Name:", champion.name)
print("Champion Version:", champion.version)
print("Run ID:", champion.run_id)

### Model Serving

- Model Serving means making your trained ML model available so that new data can be sent to the model and it returns predictions.

Simple definition

- Model Serving = Deploying a trained ML model as an endpoint/API so that users or applications can send new data and receive predictions.

#### Step 1 — Open Model Serving

In [0]:
''''
Endpoint
└── retail-sales-xgboost-endpoint

Served Entity
└── retail_project.gold.retail_sales_xgboost
    └── Version 1
        └── Traffic: 100%

Compute
└── CPU
    └── 0–4 concurrency

Scale to zero
└── ON

Tracing
└── OFF

Route optimization
└── OFF

Model Deployed Successfully.

One important correction before we proceed: because your model was logged with an input example containing the 42-element feature vector, the serving endpoint currently expects that model input. Databricks supports querying custom-model endpoints with structured scoring payloads, and the Query UI also generates example requests

#### Step 1 — Don't manually create the 42 values

In [0]:
display(
    processed_test
    .select("features", "Total_revenue")
    .limit(1)
)

#### Step 2 — Get the actual feature vector

    Real row
    ↓
    features = 42 values
    ↓
    Actual Total_revenue = target

#### Step 3 — Convert the Spark vector to a Python list

In [0]:
row = processed_test.select("features", "Total_revenue").first()

feature_values = row["features"].toArray().tolist()
actual_revenue = row["Total_revenue"]

print("Number of features:", len(feature_values))
print("Actual revenue:", actual_revenue)
print("Features:", feature_values)

### 🚀 Next step: Test the endpoint programmatically

- We shouldn't stop at the browser's **Query Endpoint test**. In a real ML project, the next step is to ca**ll the deployed model** from Python.

#### Step 1 — Python API inference

In [0]:
# Checking / verifying the model schema 

import mlflow

model_uri = "models:/retail_project.gold.retail_sales_xgboost/1"

model_info = mlflow.models.get_model_info(model_uri)

print(model_info.signature)

- Perfect. ✅ Now we have the exact model signature, so we don't need to guess the request format.

#### Step 1 — Use your real 42 features

In [0]:
features = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0,
            0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0,
            1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
            1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0,
            0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
            1.0, 1.0, 3.0, 6.0, 17.0, 1.0, 9974.0]

In [0]:
print(len(features))

#### Step 2 — Create the payload
- Because your signature is a tensor, use this format:

In [0]:
payload = {
    "inputs": [features]
}

#### Step 3 — Send it to the endpoint

In [0]:
import requests

DATABRICKS_HOST = "https://dbc-5e8b0aae-51ef.cloud.databricks.com"

ENDPOINT_NAME = "retail-sales-xgboost-endpoint"

url = f"{DATABRICKS_HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations"

print(url)


#### Step 1 — Create the Databricks token

- "How does an external Python application securely call my deployed Databricks ML model?"

In [0]:
import os

os.environ["DATABRICKS_TOKEN"] = "<YOUR_DATABRICKS_TOKEN>"

In [0]:
#2. Then retrieve it
token = os.environ["DATABRICKS_TOKEN"]

In [0]:
#3. Create the authentication header

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

In [0]:
# 4. Then use your endpoint

import requests

DATABRICKS_HOST = "https://dbc-5e8b0aae-51ef.cloud.databricks.com"
ENDPOINT_NAME = "retail-sales-xgboost-endpoint"

url = f"{DATABRICKS_HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations"
print(url)

In [0]:
# 5. Send the request 🚀

response = requests.post(
    url,
    headers=headers,
    json=payload
)

print("Status code:", response.status_code)
print("Response:", response.json())

### Next: Make a reusable prediction function
- Instead of writing the requests.post() code every time, let's create a function.

In [0]:
def predict_revenue(features):

    payload = {
        "inputs": [features]
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    response.raise_for_status()

    return response.json()["predictions"][0]

In [0]:
prediction = predict_revenue(features)

print("Predicted Revenue:", prediction)

In [0]:
#Why raise_for_status()?
print(response.raise_for_status())

### 📌 Overall Project Insights — Retail Sales ML Project

1. **Built an end-to-end retail sales ML project in Databricks**, covering data preparation, EDA, feature engineering, model training, evaluation, experiment tracking, model registry, and model serving.

2. **Implemented Medallion Architecture** using Raw → Bronze → Silver → Gold layers to organize and prepare the retail sales data for analytics and machine learning.

3. **Performed data cleaning and EDA** to understand revenue patterns across categories, products, gender, cities, states, and time-based dimensions such as month and quarter.

4. **Created ML-ready features** by handling categorical variables, creating time-based features such as Month, Quarter, WeekOfYear, DayOfWeek, and DayOfMonth, and assembling the features into a **42-dimensional feature vector** using Spark ML.

5. **Compared multiple regression models** including Linear Regression, Decision Tree, Random Forest, and XGBoost to identify the most suitable model for predicting `Total_revenue`.

6. **XGBoost achieved the best overall performance** among the evaluated models, with RMSE ≈ **4,556**, MAE ≈ **3,234**, and R² ≈ **0.9938** in the recorded evaluation.

7. **Performed multiple XGBoost experiments** by changing model configurations and found that Experiment 4 achieved the best recorded performance, with RMSE ≈ **4,481**, MAE ≈ **3,218**, and R² ≈ **0.9940**.

8. **Implemented MLflow experiment tracking** to record model parameters, metrics, runs, and model artifacts, making model experimentation and comparison more organized and reproducible.

9. **Registered the final XGBoost model in Unity Catalog and deployed it using Databricks Model Serving**, then successfully tested the deployed endpoint through a Python REST API and received a prediction with HTTP status **200**.

10. **The project provides a strong foundation for production ML**, with future improvements including data/model monitoring, drift detection, automated retraining, better time-based validation, leakage checks, richer historical data, and adding **Databricks Genie/Agent capabilities** for natural-language business analytics.


### 🔮 Recommendations for Future Improvement

1. **Implement Model Monitoring & Drift Detection**
   Continuously monitor prediction performance, data drift, and model drift, and trigger retraining when model performance decreases.

2. **Improve Data & Feature Quality**
   Perform a detailed **data-leakage check** and create more meaningful features such as customer purchase history, discounts, product trends, seasonality, and previous-period revenue.

3. **Use Time-Based Validation**
   For future sales forecasting, use chronological train/validation/test splits instead of random splitting to better represent real-world future predictions.

4. **Automate the ML Lifecycle**
   Build an automated workflow for data preparation → feature engineering → model training → MLflow experiment tracking → model evaluation → model registry → deployment → monitoring → retraining.

5. **Add Databricks Genie & AI Capabilities**
   Integrate **Genie** to allow business users to ask natural-language questions about sales data and later explore **Agent Bricks** for building more advanced AI-powered business assistants.


### 🏆Databricks Retail Sales ML Project — Overall Architecture



                    Retail Sales Data
                           │
                           ▼
                  Medallion Architecture
                           │
              ┌────────────┼────────────┐
              ▼            ▼            ▼
             Raw         Bronze       Silver
                                        │
                                        ▼
                                       Gold
                                        │
                                        ▼
                              EDA + Data Cleaning
                                        │
                                        ▼
                              Feature Engineering
                                        │
                                        ▼
                                Train / Test Data
                                        │
              ┌─────────────┬───────────┼─────────────┐
              ▼             ▼           ▼             ▼
          Linear Reg.   Decision Tree Random Forest XGBoost
                                                      │
                                                      ▼
                                             XGBoost Experiments
                                                      │
                                                      ▼
                                                 MLflow
                                                      │
                                                      ▼
                                             Model Registry
                                                      │
                                                      ▼
                                             Model Versioning
                                                      │
                                                      ▼
                                             Model Serving
                                                      │
                                                      ▼
                                                REST API
                                                      │
                                                      ▼
                                                 Prediction

### 📝 Conclusion

- This project successfully demonstrates an **end-to-end retail sales machine learning workflow using Databricks**. The project covered data preparation, Medallion Architecture, EDA, feature engineering, multiple regression models, XGBoost experimentation, MLflow experiment tracking, Unity Catalog Model Registry, and Model Serving. Among the evaluated models, **XGBoost delivered the strongest performance**, achieving an R² of approximately **0.994**. The final model was successfully registered and deployed through Databricks Model Serving, and its REST API was successfully tested. Overall, this project provided practical experience in building, tracking, managing, and deploying an ML model in a Databricks environment and provides a strong foundation for future improvements such as monitoring, automated retraining, Genie, and AI-powered analytics.
